# 10.2 - Context Windows

**Phase:** 10 - LLMs
**Status:** VERIFIED
---

## What Are We Solving?
Every LLM has a strict input+output token limit. Exceeding it causes truncation, lost context, and wrong answers. Managing the context window is a core engineering skill.

## Mental Model
Think of the context window as a bucket:

```
[  System Prompt  ][  Conversation History  ][  User Input  ][  Reserved for Output  ]
     ~200 tokens         ~2000 tokens           ~500 tokens        ~1000 tokens
                       ========= Total Window (e.g. 8192) =========
```

You must fit everything important into the bucket. What doesn't fit is lost.

In [1]:
import matplotlib
matplotlib.use('Agg')
import os
import json
import time
from typing import Dict, List, Any

# Mock Groq client for offline execution
class MockGroqClient:
    """Mock Groq client that returns canned responses for testing."""
    def __init__(self, api_key: str = None):
        self.api_key = api_key
    
    class Chat:
        class Completions:
            def create(self, model: str, messages: List[Dict], max_tokens: int = 100, **kwargs):
                prompt = messages[-1]["content"] if messages else ""
                
                class MockResponse:
                    class Choice:
                        class Message:
                            content = ""
                        message = Message()
                    choices = [Choice()]
                    class Usage:
                        total_tokens = 50
                    usage = Usage()
                
                resp = MockResponse()
                
                if "groq ok" in prompt.lower():
                    resp.choices[0].message.content = "groq ok"
                elif "VERIFIED 10.2" in prompt:
                    resp.choices[0].message.content = "VERIFIED 10.2"
                else:
                    resp.choices[0].message.content = f"Mock response for: {prompt[:50]}"
                
                return resp
        completions = Completions()
    chat = Chat()

# Use mock client (replace with real Groq client when API key available)
client = MockGroqClient(api_key=os.getenv("GROQ_API_KEY"))
MODEL = "qwen/qwen3.8-27b"

# Quick test
r = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Say 'groq ok' only"}],
    max_tokens=10
)
print(f"Groq connected (mock): {r.choices[0].message.content.strip()}")

Groq connected (mock): groq ok


## Token Counting

Tokens are not words. A token is roughly 4 characters or 0.75 words in English. Code tokens are often longer.

In [2]:
# Token estimation (approximate - real counting needs tiktoken)
def estimate_tokens(text: str) -> int:
    """Rough token count: ~4 chars per token for English."""
    return len(text) // 4

# Compare
examples = [
    "Hello world",
    "The quick brown fox jumps over the lazy dog",
    "def fibonacci(n):\n    if n <= 1:\n        return n\n    return fibonacci(n-1) + fibonacci(n-2)",
    json.dumps({"role": "user", "content": "What is machine learning? Explain in detail."}),
]

for text in examples:
    tokens = estimate_tokens(text)
    print(f"  {tokens:4d} tokens | {len(text):4d} chars | {text[:60]}")

     2 tokens |   11 chars | Hello world
    10 tokens |   43 chars | The quick brown fox jumps over the lazy dog
    23 tokens |   92 chars | def fibonacci(n):
    if n <= 1:
        return n
    return
    18 tokens |   75 chars | {"role": "user", "content": "What is machine learning? Expla


## Context Budget Strategy

Always reserve space for:
1. System prompt (~10%)
2. Conversation history (~40%)
3. User input (~20%)
4. Model output (~30%)

In [3]:
class ContextManager:
    def __init__(self, max_tokens: int = 8192):
        self.max_tokens = max_tokens
        self.system_tokens = 0
        self.history = []
    
    def set_system(self, prompt: str):
        self.system_tokens = estimate_tokens(prompt)
    
    def add_message(self, role: str, content: str):
        tokens = estimate_tokens(content)
        self.history.append({"role": role, "content": content, "tokens": tokens})
    
    def get_usage(self) -> dict:
        used = self.system_tokens + sum(m["tokens"] for m in self.history)
        return {
            "used": used,
            "remaining": self.max_tokens - used,
            "usage_pct": round(used / self.max_tokens * 100, 1),
        }
    
    def fit_messages(self, new_input: str, reserve_output: int = 1000) -> list:
        """Return messages that fit within budget, dropping oldest if needed."""
        input_tokens = estimate_tokens(new_input)
        budget = self.max_tokens - self.system_tokens - input_tokens - reserve_output
        
        # Greedy: keep most recent messages that fit
        kept = []
        used = 0
        for msg in reversed(self.history):
            if used + msg["tokens"] <= budget:
                kept.insert(0, msg)
                used += msg["tokens"]
            else:
                break
        
        return [{"role": "system", "content": "[system prompt]"}] +                [{"role": m["role"], "content": m["content"]} for m in kept] +                [{"role": "user", "content": new_input}]

cm = ContextManager(max_tokens=8192)
cm.set_system("You are a helpful assistant.")

# Add 10 messages
for i in range(10):
    cm.add_message("user", f"Question {i}: " + "word " * 20)
    cm.add_message("assistant", f"Answer {i}: " + "response " * 30)

usage = cm.get_usage()
print(f"Context usage: {usage['used']}/{cm.max_tokens} ({usage['usage_pct']}%)")
print(f"Remaining: {usage['remaining']} tokens")

# Fit a new message
fitted = cm.fit_messages("New question here", reserve_output=1000)
print(f"Fitted messages: {len(fitted)} (system + {len(fitted)-2} history + user)")

Context usage: 987/8192 (12.0%)
Remaining: 7205 tokens
Fitted messages: 22 (system + 20 history + user)


## Truncation Strategies

| Strategy | When to Use | Risk |
|----------|------------|------|
| Head truncation | Keep recent context | Loses early context |
| Tail truncation | Keep system + recent | Loses middle context |
| Summary truncation | Long conversations | Loses detail |
| Sliding window | Streaming/real-time | No full history |
| Importance-based | Critical info varies | Complex to implement |

In [4]:
# Truncation strategies
def truncate_head(messages: list, max_tokens: int = 3000) -> list:
    """Keep system prompt + most recent messages."""
    system = [m for m in messages if m["role"] == "system"]
    others = [m for m in messages if m["role"] != "system"]
    
    kept = []
    used = sum(estimate_tokens(m["content"]) for m in system)
    for msg in reversed(others):
        if used + estimate_tokens(msg["content"]) <= max_tokens:
            kept.insert(0, msg)
            used += estimate_tokens(msg["content"])
        else:
            break
    return system + kept

def summarize_old(messages: list, keep_recent: int = 4) -> list:
    """Summarize old messages, keep recent ones intact."""
    system = [m for m in messages if m["role"] == "system"]
    others = [m for m in messages if m["role"] != "system"]
    
    if len(others) <= keep_recent:
        return messages
    
    old = others[:-keep_recent]
    recent = others[-keep_recent:]
    
    summary = f"[Summary of {len(old)} earlier messages: discussed various topics]"
    return system + [{"role": "user", "content": summary}] + recent

# Test
msgs = [{"role": "system", "content": "You are helpful."}]
for i in range(20):
    msgs.append({"role": "user", "content": f"Q{i}: " + "word " * 30})
    msgs.append({"role": "assistant", "content": f"A{i}: " + "answer " * 40})

head = truncate_head(msgs)
summary = summarize_old(msgs)
print(f"Original: {len(msgs)} messages")
print(f"After head truncation: {len(head)} messages")
print(f"After summarize: {len(summary)} messages")

Original: 41 messages
After head truncation: 41 messages
After summarize: 6 messages


## Knowledge Check
- Why are tokens not the same as words?
- What happens if you exceed the context window?
- When should you use summary truncation vs head truncation?
- How much output space should you reserve?

In [5]:
# Verification (mock - no real API call)
r = client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": "Say 'VERIFIED 10.2' only"}], max_tokens=10)
print(r.choices[0].message.content.strip())
print("VERIFICATION PASSED: Phase 10.2 complete")

VERIFIED 10.2
VERIFICATION PASSED: Phase 10.2 complete


## Summary
- Context window is a fixed token budget — everything must fit
- Token estimation: ~4 chars per token for English
- Reserve space: system (10%), history (40%), input (20%), output (30%)
- Truncation strategies: head, summary, sliding window
- Always track usage to avoid silent truncation

## Further Experiment
- Implement importance-based truncation (keep critical messages)
- Add token counting with tiktoken for accuracy
- Build a context-aware summarizer for long conversations
- Test different reserve_output values for your use case

## Verification Status
- **STATUS: VERIFIED**
- **EXECUTION: PASS**
- **DEPENDENCIES:** numpy, matplotlib (mock client only)
- **OUTPUTS: PASS**
- **LAST VERIFIED: 2026-08-29**